In [1]:
import pandas as pd
import utils as uti
import matplotlib.pyplot as plt

In [2]:
df = pd.read_pickle("../data/train_df.pkl")

In [3]:
import pickle
from pathlib import Path

file_path = Path("../data/models.pkl")

if file_path.exists():
    print('loading model file')
    with file_path.open("rb") as file:
        models = pickle.load(file)
else:
    print('creating new model file')
    models = {}

loading model file


# param optimization

In [ ]:
walds = pd.read_pickle("../data/walds.pkl")
walds.head()

In [ ]:
X = df[walds['var'].tolist()]
y = df.Class

In [ ]:
import numpy as np
import optuna
import xgboost as xgb

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score


def objective(trial):
    params = {
        "objective": "binary:logistic",
        "eval_metric": "aucpr",
        "tree_method": "hist",

        "n_estimators": trial.suggest_int(
            "n_estimators", 200, 2000
        ),
        "max_depth": trial.suggest_int(
            "max_depth", 2, 10
        ),
        "learning_rate": trial.suggest_float(
            "learning_rate", 0.005, 0.2, log=True
        ),
        "min_child_weight": trial.suggest_float(
            "min_child_weight", 1, 20, log=True
        ),
        "subsample": trial.suggest_float(
            "subsample", 0.6, 1.0
        ),
        "colsample_bytree": trial.suggest_float(
            "colsample_bytree", 0.6, 1.0
        ),
        "gamma": trial.suggest_float(
            "gamma", 0, 10
        ),
        "reg_alpha": trial.suggest_float(
            "reg_alpha", 1e-8, 10, log=True
        ),
        "reg_lambda": trial.suggest_float(
            "reg_lambda", 1e-8, 10, log=True
        ),

        "random_state": 42,
        "n_jobs": -1,
    }

    cv = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    fold_pr_aucs = []

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y)):
        X_train = X.iloc[train_idx]
        X_valid = X.iloc[valid_idx]
        y_train = y.iloc[train_idx]
        y_valid = y.iloc[valid_idx]

        model = xgb.XGBClassifier(**params)

        model.fit(
            X_train,
            y_train,
            eval_set=[(X_valid, y_valid)],
            verbose=False
        )

        predictions = model.predict_proba(X_valid)[:, 1]

        # Average precision is commonly used as the PR-AUC summary metric
        pr_auc = average_precision_score(
            y_valid,
            predictions
        )

        fold_pr_aucs.append(pr_auc)

        # Report the mean PR-AUC achieved so far
        trial.report(
            np.mean(fold_pr_aucs),
            step=fold
        )

        if trial.should_prune():
            raise optuna.TrialPruned()

    return np.mean(fold_pr_aucs)


In [ ]:
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(
        n_startup_trials=10,
        n_warmup_steps=2
    )
)

study.optimize(
    objective,
    n_trials=25,
    show_progress_bar=True
)

print("Best AUC:", study.best_value)
print("Best parameters:")
print(study.best_params)


In [5]:
import optuna
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, classification_report

In [9]:
X = df.drop(columns=["Class"])
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [10]:
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int(
            "n_estimators", 50, 300
        ),

        # Include shallow trees; avoid allowing only very flexible models
        "max_depth": trial.suggest_categorical(
            "max_depth", [3, 4, 5, 6, 8, 12, None]
        ),

        "min_samples_split": trial.suggest_int(
            "min_samples_split", 5, 100
        ),

        "min_samples_leaf": trial.suggest_int(
            "min_samples_leaf", 2, 50
        ),

        "max_features": trial.suggest_categorical(
            "max_features", ["sqrt", "log2", 0.3, 0.5, None]
        ),

        "max_samples": trial.suggest_float(
            "max_samples", 0.5, 1.0
        ),

        "class_weight": trial.suggest_categorical(
            "class_weight",
            [None, "balanced", "balanced_subsample"]
        ),

        "random_state": 42,
        "n_jobs": 1,
    }
        
    # {
    #     "n_estimators": trial.suggest_int(
    #         "n_estimators", 50, 300
    #     ),
    #     "max_depth": trial.suggest_int(
    #         "max_depth", 3, 20
    #     ),
    #     "min_samples_split": trial.suggest_int(
    #         "min_samples_split", 2, 20
    #     ),
    #     "min_samples_leaf": trial.suggest_int(
    #         "min_samples_leaf", 1, 10
    #     ),
    #     "max_features": trial.suggest_categorical(
    #         "max_features", ["sqrt", "log2", None]
    #     ),
    #     "class_weight": trial.suggest_categorical(
    #         "class_weight",
    #         ["balanced", "balanced_subsample"]
    #     ),
    #     "random_state": 42,
    #     "n_jobs": 1,
    # }

    model = RandomForestClassifier(**params)

    cv = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="average_precision",
        n_jobs=-1
    )

    return scores.mean()


In [ ]:
study = optuna.create_study(
    direction="maximize",
    study_name="random_forest_classifier"
)

study.optimize(objective, n_trials=15, show_progress_bar=True)

print("Best CV PR AUC:", study.best_value)
print("Best parameters:")
print(study.best_params)

[I 2026-09-09 00:04:55,554] A new study created in memory with name: random_forest_classifier


  0%|          | 0/15 [00:00<?, ?it/s]

## optimization summary

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
fig = optuna.visualization.matplotlib.plot_optimization_history(study)
plt.title("Optuna Optimization History")
plt.tight_layout()
plt.show()

In [ ]:
from optuna.importance import get_param_importances

importance_dict = get_param_importances(study)

In [ ]:
fig = optuna.visualization.matplotlib.plot_param_importances(study)
plt.title("Hyperparameter Importance")
plt.tight_layout()
plt.show()

In [ ]:
fig = optuna.visualization.matplotlib.plot_slice(
    study,
    params=list(importance_dict.keys())
)
plt.tight_layout()
plt.show()

In [ ]:
fig = optuna.visualization.matplotlib.plot_parallel_coordinate(
    study,
    params=list(importance_dict.keys())
)
plt.tight_layout()
plt.show()

In [ ]:
fig = optuna.visualization.matplotlib.plot_contour(
    study,
    params=list(importance_dict.keys())[:2]
)
plt.tight_layout()
plt.show()

In [ ]:
trials_df = study.trials_dataframe()

best_trials = (
    trials_df
    .query("state == 'COMPLETE'")
    .sort_values("value", ascending=False)
    .head(10)
)

best_trials

# fit best model

In [ ]:
best_model = RandomForestClassifier(
    **study.best_params,
    random_state=42,
    n_jobs=-1
)

best_model.fit(X_train, y_train)

In [ ]:
models['rf'] = best_model

## partial dependence plots

In [ ]:
# from sklearn.inspection import PartialDependenceDisplay

# PartialDependenceDisplay.from_estimator(
#     best_model,
#     X,
#     features=walds['var'].tolist()
# )

# plt.show()

# save model

In [ ]:
with open("../data/models.pkl", "wb") as file:
    pickle.dump(models, file)